# Module 3 — Intent Classifier

**Part of the RAG-Based E-commerce Customer Support Chatbot**

**Stage 3** of the pipeline: routes each message into one of 7 macro-intent categories, which
the Flask app then uses to decide whether to answer directly, escalate, or invoke the RAG
pipeline.

**Dataset:** [`bitext/Bitext-customer-support-llm-chatbot-training-dataset`](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset)
— ~27k rows, with an `intent` column containing 27 fine-grained intents and a `category` column
grouping them further.

**Macro-category mapping used below (27 → 7):**

| Macro category | Bitext intents mapped in |
|---|---|
| `order_status` | track_order, track_refund, delivery_options, delivery_period |
| `order_management` | place_order, cancel_order, change_order, change_shipping_address, set_up_shipping_address |
| `billing_and_refunds` | check_invoice, get_invoice, get_refund, check_refund_policy, payment_issue, check_payment_methods, check_cancellation_fee |
| `account_management` | create_account, delete_account, edit_account, switch_account, recover_password, registration_problems, newsletter_subscription |
| `complaint` | complaint, review |
| `out_of_scope` | contact_customer_service, contact_human_agent |
| `greeting / goodbye / gratitude` | *(none — see note below)* |

> **⚠️ Important data note:** the Bitext dataset's 27 intents are all *task* intents — it does
> not contain any labeled examples of pure greetings, goodbyes, or thank-yous. That means the
> classifier trained below can only ever predict the other 6 macro-categories; it has literally
> never seen a `greeting/goodbye/gratitude` example, so it would be dishonest to claim it can
> classify that category from learned weights.
>
> To still satisfy the routing requirement, **Notebook 5 (Flask app) handles
> `greeting/goodbye/gratitude` with a fast, transparent keyword/regex pre-filter that runs
> *before* this classifier is called** (e.g. matching "hi", "hello", "thanks", "bye", etc.).
> Only messages that don't match that pre-filter are sent to the trained 6-class model below.
> This is called out explicitly in code comments in both notebooks so it's never a hidden
> assumption.

**Model:** TF-IDF (word n-grams) + `LinearSVC` — a strong, fast, interpretable baseline for
short-text intent classification; well-suited to a dataset of this size.

**Output artifacts:** `intent_tfidf_vectorizer.joblib`, `intent_classifier.joblib`.


## 1. Install dependencies

In [ ]:
!pip install -q datasets scikit-learn joblib pandas


## 2. Imports

In [ ]:
import re
import joblib
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

RANDOM_STATE = 42


## 3. Load the dataset

In [ ]:
raw = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
print(raw)

df = raw["train"].to_pandas()
print(df.shape)
print(df.columns.tolist())
df.head()


README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})
(26872, 5)
['flags', 'instruction', 'category', 'intent', 'response']


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [ ]:
print("Unique intents:", df['intent'].nunique())
print(sorted(df['intent'].unique()))
print()
print(df['intent'].value_counts())


Unique intents: 27
['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoice', 'check_payment_methods', 'check_refund_policy', 'complaint', 'contact_customer_service', 'contact_human_agent', 'create_account', 'delete_account', 'delivery_options', 'delivery_period', 'edit_account', 'get_invoice', 'get_refund', 'newsletter_subscription', 'payment_issue', 'place_order', 'recover_password', 'registration_problems', 'review', 'set_up_shipping_address', 'switch_account', 'track_order', 'track_refund']

intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent          999
check_payment_methods        999
delivery_period              999
newsletter_subscription      999
get_invoice                  999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order        

## 4. Map 27 fine-grained intents -> 6 trainable macro-categories

In [ ]:
INTENT_TO_MACRO = {
    # order_status
    "track_order": "order_status",
    "track_refund": "order_status",
    "delivery_options": "order_status",
    "delivery_period": "order_status",
    # order_management
    "place_order": "order_management",
    "cancel_order": "order_management",
    "change_order": "order_management",
    "change_shipping_address": "order_management",
    "set_up_shipping_address": "order_management",
    # billing_and_refunds
    "check_invoice": "billing_and_refunds",
    "get_invoice": "billing_and_refunds",
    "get_refund": "billing_and_refunds",
    "check_refund_policy": "billing_and_refunds",
    "payment_issue": "billing_and_refunds",
    "check_payment_methods": "billing_and_refunds",
    "check_cancellation_fee": "billing_and_refunds",
    # account_management
    "create_account": "account_management",
    "delete_account": "account_management",
    "edit_account": "account_management",
    "switch_account": "account_management",
    "recover_password": "account_management",
    "registration_problems": "account_management",
    "newsletter_subscription": "account_management",
    # complaint
    "complaint": "complaint",
    "review": "complaint",
    # out_of_scope (routed to human / general contact, not covered by the automated macro flows)
    "contact_customer_service": "out_of_scope",
    "contact_human_agent": "out_of_scope",
}

df["macro_intent"] = df["intent"].map(INTENT_TO_MACRO)

unmapped = df[df["macro_intent"].isna()]
if len(unmapped) > 0:
    print("WARNING: unmapped intents found:", unmapped["intent"].unique())
else:
    print("All 27 intents mapped successfully.")

print()
print(df["macro_intent"].value_counts())


All 27 intents mapped successfully.

macro_intent
account_management     6985
billing_and_refunds    6941
order_management       4963
order_status           3987
out_of_scope           1999
complaint              1997
Name: count, dtype: int64


## 5. Preprocessing + train/val/test split

In [ ]:
def basic_clean(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["instruction"].apply(basic_clean)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df["macro_intent"]
)
valid_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=RANDOM_STATE, stratify=temp_df["macro_intent"]
)

print(train_df.shape, valid_df.shape, test_df.shape)


(21497, 7) (2687, 7) (2688, 7)


## 6. TF-IDF vectorization (word n-grams)

In [ ]:
vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=20000,
    sublinear_tf=True,
    min_df=2,
)

X_train = vectorizer.fit_transform(train_df["clean_text"])
X_valid = vectorizer.transform(valid_df["clean_text"])
X_test = vectorizer.transform(test_df["clean_text"])

y_train = train_df["macro_intent"].values
y_valid = valid_df["macro_intent"].values
y_test = test_df["macro_intent"].values

print("TF-IDF matrix shape (train):", X_train.shape)


TF-IDF matrix shape (train): (21497, 5180)


## 7. Train the classifier

`LinearSVC` doesn't natively output probabilities, so we wrap it in `CalibratedClassifierCV`
to get well-calibrated confidence scores for the inference function (useful for the Flask app
to decide e.g. "low confidence -> escalate to human").


In [ ]:
base_clf = LinearSVC(C=1.0, random_state=RANDOM_STATE, max_iter=5000)
clf = CalibratedClassifierCV(base_clf, method="sigmoid", cv=3)
clf.fit(X_train, y_train)
print("Training complete.")


Training complete.


## 8. Evaluate

In [ ]:
def evaluate(name, X, y_true):
    y_pred = clf.predict(X)
    acc = accuracy_score(y_true, y_pred)
    print(f"=== {name} accuracy: {acc:.4f} ===")
    print(classification_report(y_true, y_pred, zero_division=0))
    return acc

_ = evaluate("Validation", X_valid, y_valid)
test_acc = evaluate("Test", X_test, y_test)


=== Validation accuracy: 0.9978 ===
                     precision    recall  f1-score   support

 account_management       1.00      1.00      1.00       698
billing_and_refunds       1.00      1.00      1.00       694
          complaint       1.00      1.00      1.00       200
   order_management       0.99      1.00      1.00       496
       order_status       1.00      0.99      1.00       399
       out_of_scope       1.00      1.00      1.00       200

           accuracy                           1.00      2687
          macro avg       1.00      1.00      1.00      2687
       weighted avg       1.00      1.00      1.00      2687

=== Test accuracy: 1.0000 ===
                     precision    recall  f1-score   support

 account_management       1.00      1.00      1.00       699
billing_and_refunds       1.00      1.00      1.00       694
          complaint       1.00      1.00      1.00       199
   order_management       1.00      1.00      1.00       497
       order_st

## 9. Reusable inference function

`classify_intent(text)` includes the keyword pre-filter for `greeting / goodbye / gratitude`
described above (since the trained model was never shown examples of that class), then falls
back to the trained classifier for everything else.


In [ ]:
GREETING_PATTERNS = re.compile(
    r"^\s*("
    r"hi|hello|hey|good (morning|afternoon|evening)|"
    r"bye|goodbye|see you|take care|"
    r"thanks|thank you|thx|appreciate it|much appreciated"
    r")\b",
    re.IGNORECASE,
)

def classify_intent(text: str) -> dict:
    """Route a customer message into one of 7 macro-intent categories.

    Note: 'greeting / goodbye / gratitude' is detected via a keyword pre-filter, not the
    trained model, because the Bitext dataset contains no examples of that category
    (see the data note in this notebook's introduction).

    Args:
        text: raw customer message.

    Returns:
        {"intent": <str>, "confidence": <float 0-1>, "source": "rule" | "model"}
    """
    if not text or not text.strip():
        return {"intent": "out_of_scope", "confidence": 0.0, "source": "rule"}

    stripped = text.strip()
    if GREETING_PATTERNS.match(stripped) and len(stripped.split()) <= 6:
        return {"intent": "greeting/goodbye/gratitude", "confidence": 1.0, "source": "rule"}

    cleaned = basic_clean(text)
    vec = vectorizer.transform([cleaned])
    proba = clf.predict_proba(vec)[0]
    pred_idx = int(np.argmax(proba))
    pred_label = clf.classes_[pred_idx]
    return {
        "intent": pred_label,
        "confidence": round(float(proba[pred_idx]), 4),
        "source": "model",
    }


# Quick smoke test
for sample in [
    "Hi there!",
    "Thanks a lot, bye!",
    "Where is my order #4821?",
    "I want to cancel my subscription and delete my account",
    "This product broke after one day, I'm very disappointed",
    "I need to speak to a real person right now",
]:
    print(sample, "->", classify_intent(sample))


Hi there! -> {'intent': 'greeting/goodbye/gratitude', 'confidence': 1.0, 'source': 'rule'}
Thanks a lot, bye! -> {'intent': 'greeting/goodbye/gratitude', 'confidence': 1.0, 'source': 'rule'}
Where is my order #4821? -> {'intent': 'order_status', 'confidence': 0.9911, 'source': 'model'}
I want to cancel my subscription and delete my account -> {'intent': 'account_management', 'confidence': 0.9992, 'source': 'model'}
This product broke after one day, I'm very disappointed -> {'intent': 'order_management', 'confidence': 0.859, 'source': 'model'}
I need to speak to a real person right now -> {'intent': 'out_of_scope', 'confidence': 0.9993, 'source': 'model'}


## 10. Mount Google Drive and set up the project folder structure

All 4 notebooks share one Drive folder, `RAG_chatbot_project/`, organized into one subfolder
per module so artifacts never collide and `app.py` can load each module from a predictable
path:

```
RAG_chatbot_project/
├── language_detection/      <- Notebook 1 saves here
├── sentiment_classifier/    <- Notebook 2 saves here
├── intent_classifier/       <- this notebook saves here
└── rag_pipeline/            <- Notebook 4 saves here
```

This cell mounts Drive and creates the full structure (safe to re-run).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/RAG_chatbot_project"
LANGUAGE_DIR = os.path.join(BASE_DIR, "language_detection")
SENTIMENT_DIR = os.path.join(BASE_DIR, "sentiment_classifier")
INTENT_DIR = os.path.join(BASE_DIR, "intent_classifier")
RAG_DIR = os.path.join(BASE_DIR, "rag_pipeline")

for d in [LANGUAGE_DIR, SENTIMENT_DIR, INTENT_DIR, RAG_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted. Project folder structure ready under:", BASE_DIR)


Mounted at /content/drive
Drive mounted. Project folder structure ready under: /content/drive/MyDrive/RAG_chatbot_project


## 11. Save artifacts

In [ ]:
joblib.dump(vectorizer, os.path.join(INTENT_DIR, "intent_tfidf_vectorizer.joblib"))
joblib.dump(clf, os.path.join(INTENT_DIR, "intent_classifier.joblib"))

print("Saved intent classifier artifacts to:", INTENT_DIR)
print(f"Final test accuracy: {test_acc:.4f}")


Saved intent classifier artifacts to: /content/drive/MyDrive/RAG_chatbot_project/intent_classifier
Final test accuracy: 1.0000
